# Ticekt DENG-008: Create Bronze Delta Tables with Auto Loader
**Context**: Write two Auto Loader streams in a Databricks notebook - one ingesting 
`match_history/` JSON files into `marvel_rivals.bronze.raw_match_history`, and one ingesting `match_details/` JSON files into `marvel_rivals.bronze.raw_match_details`.


In [0]:
MATCH_HISTORY_PATH = "/Volumes/marvel_rivals/bronze/raw_files/match_history/"
MATCH_DETAILS_PATH = "/Volumes/marvel_rivals/bronze/raw_files/match_details/"

CHECKPOINT_HISTORY = "/Volumes/marvel_rivals/bronze/checkpoints/match_history/"
CHECKPOINT_DETAILS = "/Volumes/marvel_rivals/bronze/checkpoints/match_details/"

TARGET_HISTORY_TABLE = "marvel_rivals.bronze.raw_match_history"
TARGET_DETAILS_TABLE = "marvel_rivals.bronze.raw_match_details"

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS marvel_rivals.bronze.checkpoints;

In [0]:
dbutils.fs.head("/Volumes/marvel_rivals/bronze/raw_files/match_history/Player_You.json", 500)

In [0]:
# Step 1: Drop the table entirely (clean slate)
spark.sql("DROP TABLE IF EXISTS marvel_rivals.bronze.raw_match_history")

# Step 2: Clear the checkpoint
dbutils.fs.rm("/Volumes/marvel_rivals/bronze/checkpoints/match_history/", recurse=True)

# Step 3: Re-run Auto Loader (single clean pass)

In [0]:
(spark.readStream
     .format("cloudFiles")                  # This tels Spark to use Auto Loader
     .option("cloudFiles.format", "json")   # our files are JSON
     .option("cloudFiles.schemaLocation", CHECKPOINT_HISTORY) # where to store inferred schema
     .load(MATCH_HISTORY_PATH)              # where to read files from / the folder to watch
     .writeStream
     .option("checkpointLocation", CHECKPOINT_HISTORY) # where to track processed files
     .option("mergeSchema", "true")         # handle Schema evolution gracefully
     .trigger(availableNow=True)            # tells auto loader to "process all currently available files, then stop"
     .toTable(TARGET_HISTORY_TABLE)         # Write this to delta table
)


In [0]:
# Row count
count = spark.table("marvel_rivals.bronze.raw_match_history").count()
print(f"Total number of records: {count}")

# Preview Schema
spark.table("marvel_rivals.bronze.raw_match_history").printSchema()

## SQL Alternatives for Match History
`COPY INTO` is SQL's answer to batch file ingestion - it tracks which files have already been loaded (similar to Auto Loader's checkpointing) and only processes new ones each run. Simpler than Auto Loader, good for batch scenarios

---

**Lakeflow Declarative Pipelines (formerly DLT)**
This is the more modern apprach - `read_files()` with `STREAM` is essentially Auto Loader expressed in SQL, and it's part of the Lakeflow Declarative Pipelines

In [0]:
%sql
-- COPY INTO (SQL-native, simpler)
COPY INTO marvel_rivals.bronze.raw_match_history
FROM '/Volumes/marvel_rivals/bronze/raw_files/match_history'
FILEFORMAT = JSON
FORMAT_OPTIONS ('inferSchema' = 'true', 'mergeSchema' = 'true')
COPY_OPTIONS ('mergeSchema' = 'true');

SELECT * FROM marvel_rivals.bronze.raw_match_history

In [0]:
%sql
CREATE or REFRESH STREAMING TABLE raw_match_history
AS SELECT * FROM STREAM read_files(
    '/Volumes/marvel_rivals/bronze/raw_files/match_history',
    format => 'json',
    inferSchema => true
);

In [0]:
dbutils.fs.head("/Volumes/marvel_rivals/bronze/raw_files/match_details/5513494_1780879146_2042395_11001_12.json", 200)

In [0]:
(
   spark.readStream
   .format("cloudFiles")
   .option("cloudFiles.format", "json")
   .option("multiline", "true")
   .option("cloudFiles.schemaLocation", CHECKPOINT_DETAILS)
   .load(MATCH_DETAILS_PATH)
   .writeStream
   .option("checkpointLocation", CHECKPOINT_DETAILS)
   .option("mergeSchema", "true")
   .trigger(availableNow=True)
   .toTable(TARGET_DETAILS_TABLE) 
)

In [0]:
# Row count
count = spark.table("marvel_rivals.bronze.raw_match_details").count()
print(f"Total number of records: {count}")

# Preview Schema
spark.table("marvel_rivals.bronze.raw_match_details").printSchema()

In [0]:
print("match_history rescued:")
spark.sql("""
        SELECT _rescued_data
        FROM marvel_rivals.bronze.raw_match_history
        WHERE _rescued_data IS NOT NULL  
""").show(5, truncate=False)

In [0]:
print("match_details rescued:")
spark.sql("""
        SELECT _rescued_data
        FROM marvel_rivals.bronze.raw_match_details
        WHERE _rescued_data IS NOT NULL  
""").show(5, truncate=False)